# Myocardial - prepare the demo tables

**Input** - the UCI [Myocardial Infarction Complications](https://archive.ics.uci.edu/dataset/579/myocardial+infarction+complications) archive, downloaded once to `data/myocardial/raw/uci579/` and read from there afterwards:

| File | |
| --- | --- |
| `MI.data` | 124 bare columns - no header, `?` for missing |
| `variables.json` | the published variable list: names, roles, descriptions |

**Output** - at the paths `configs/myocardial.yaml` declares:

| File | |
| --- | --- |
| `data/myocardial/train.csv` · `valid.csv` · `test.csv` | the fixed 60 / 20 / 20 split the pipeline reads |
| `data/myocardial/few_shot.csv` | the example rows a discovery run shows: 10 batches of 32, same columns as `train.csv` plus `batch` |
| `data/myocardial/screen_train.csv` · `screen_valid.csv` | the rows discovery's screen fits and scores on, same format as `train.csv` |
| `data/myocardial/column_mapping.csv` | sanitized name → the archive's name |
| `data/myocardial/column_descriptions.json` | each column's description and coding scheme, for discovery |

**The data** - 1,700 patients admitted with myocardial infarction, 111 clinical features, predicting chronic heart failure (ZSN) - 23.2% of patients. The hardest of the demo tables, on purpose:

- **Missingness is real.** 8.5% of feature cells, very unevenly: serum CPK is absent for 99.8% of patients. Nothing is imputed - the tables carry NaN - so the missingness gate and the missing-indicator features both do real work.
- **The file has no header.** The names come from `variables.json`, so getting them wrong is silent - every value stays a valid number under the wrong name. The structure is checked against that list rather than assumed.
- **Most columns are coded categories.** 98 of 110 features are ordinal or binary codes; only 12 are measurements, which the config declares as continuous.

**The scenario** - the incumbent model uses what routine admission records: history, examination, ECG, treatment. The candidates are the **serum laboratory panel** - potassium, sodium, AlAT, AsAT, CPK, white cell count, ESR - and two threshold flags. `gipo_k` is `k_blood < 4` and `giper_na` is `na_blood > 150`, so the family carries its own redundancy control with nothing planted.

**Leakage** - the archive has 12 outcome columns, all complications of the same infarction recorded at the same time. The 11 not modelled are dropped, and so is `ZSN_A` - heart failure *in the anamnesis*, which a patient with the outcome largely already has.

In [1]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# The repo root, wherever this notebook is run from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from preprocessing import (
    build_sample,
    build_shot_batches,
    fetch_archive,
    resolve_path,
    sanitize_columns,
    stratified_split,
    write_splits,
)
from validation import load_config
from validation.data import resolve_features

pd.set_option("display.width", 160, "display.max_columns", 12)

In [2]:
# Input: the UCI archive, cached here after the first download.
DATA_URL = ("https://archive.ics.uci.edu/static/public/579/"
            "myocardial+infarction+complications.zip")
VARS_URL = "https://archive.ics.uci.edu/api/dataset?id=579"
RAW_DIR = ROOT / "data" / "myocardial" / "raw" / "uci579"

# Output: written to the paths this config declares, and checked against it.
CONFIG = ROOT / "configs" / "myocardial.yaml"

SEED = 42                          # the split and the clustering
VALID_SIZE, TEST_SIZE = 0.2, 0.2   # 60 / 20 / 20
SHOTS, SHOT_BATCHES = 32, 10       # example rows per batch; one batch per discovery round
SCREEN_SIZE, SCREEN_BALANCE = {"train": 1000, "valid": 340}, False    # the screen rows; nearly all of a small table

cfg = load_config(CONFIG)
target, id_col = cfg.data.target, cfg.data.id_cols[0]

## 1. Load the archive

The variable list is the schema, so it is checked hard: 124 variables, the record id first, and the 12 outcome columns last in the published order. A reordering would rename every column silently. Two columns with an unmistakable real-world range are then spot-checked - this is what caught an off-by-one in the header during development.

In [3]:
OUTCOMES = [
    "FIBR_PREDS", "PREDS_TAH", "JELUD_TAH", "FIBR_JELUD", "A_V_BLOK",
    "OTEK_LANC", "RAZRIV", "DRESSLER", "ZSN", "REC_IM", "P_IM_STEN", "LET_IS",
]

payload = json.loads(fetch_archive(VARS_URL, RAW_DIR, "variables.json").read_text(encoding="utf-8"))
variables = (payload.get("data") or payload).get("variables") or []
assert len(variables) == 124, f"expected 124 published variables, got {len(variables)}"
assert variables[0]["role"] == "ID", f"expected the record id first, got {variables[0]['name']!r}"
assert [v["name"] for v in variables[-12:]] == OUTCOMES, "the published variable order has changed"

raw = pd.read_csv(fetch_archive(DATA_URL, RAW_DIR, "MI.data"), header=None, na_values="?")
assert raw.shape[1] == len(variables), f"MI.data has {raw.shape[1]} columns, the list {len(variables)}"
raw.columns = [v["name"] for v in variables]

for column, (low, high) in {"AGE": (18, 120), "NA_BLOOD": (100, 200)}.items():
    observed = raw[column].dropna()
    assert observed.min() >= low and observed.max() <= high, (
        f"{column} is outside [{low}, {high}] - the names are probably misaligned")

print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.iloc[:5, :8]

1,700 rows x 124 columns


,ID,AGE,SEX,INF_ANAM,STENOK_AN,FK_STENOK,IBS_POST,IBS_NASL
0,1,77.0,1,2.0,1.0,1.0,2.0,NaN
1,2,55.0,1,1.0,0.0,0.0,0.0,0.0
2,3,52.0,1,0.0,0.0,0.0,2.0,NaN
3,4,68.0,0,0.0,0.0,0.0,2.0,NaN
4,5,60.0,1,0.0,0.0,0.0,2.0,NaN


## 2. Shape the table

Drop the 11 outcomes not modelled and the leaky `ZSN_A`, sanitize the names, and rename `ZSN` to the config's target. The archive's own `ID` is the id column. Each description keeps its full coding scheme ("0: none, 1: I FC, ...") on one line - that is what tells a proposer a column is a grade and not a count.

In [4]:
frame = raw.drop(columns=[c for c in OUTCOMES if c != "ZSN"] + ["ZSN_A"])
frame, mapping = sanitize_columns(frame)
frame = frame.rename(columns={"zsn": target})
mapping.loc[mapping["column"] == "zsn", "column"] = target

descriptions = {
    v["name"].lower(): re.sub(r"\s+", " ", str(v["description"])).strip()
    + (f" Units: {v['units']}." if v.get("units") else "")
    for v in variables
}
descriptions = {k: v for k, v in descriptions.items() if k in frame.columns}
frame.shape

(1700, 112)

## 3. Profile

Missingness is the defining property of this table.

In [5]:
features = [c for c in frame.columns if c not in {target, id_col}]
profile = pd.DataFrame({
    "dtype": frame[features].dtypes.astype(str),
    "n_unique": frame[features].nunique(),
    "missing_rate": frame[features].isna().mean().round(4),
})
print(f"{len(frame):,} rows, {len(features)} features")
print(f"target {target!r}: {frame[target].mean():.2%} positive "
      f"({int(frame[target].sum()):,} of {len(frame):,})")
print(f"missing: {frame[features].isna().to_numpy().mean():.1%} of feature cells, "
      f"{int((profile['missing_rate'] > 0).sum())} of {len(features)} columns affected")
profile.sort_values("missing_rate", ascending=False).head(8)

1,700 rows, 110 features
target 'chf': 23.18% positive (394 of 1,700)
missing: 8.5% of feature cells, 109 of 110 columns affected


,dtype,n_unique,missing_rate
kfk_blood,float64,4,0.9976
ibs_nasl,float64,2,0.9576
s_ad_kbrig,float64,30,0.6329
d_ad_kbrig,float64,21,0.6329
not_na_kb,float64,2,0.4035
lid_kb,float64,2,0.3982
na_kb,float64,2,0.3865
giper_na,float64,2,0.2206


## 4. Split 60 / 20 / 20

Stratified on the target, so each part keeps the base rate, with a fixed seed. This is the **only** split: the pipeline reads these three tables exactly as written and never re-splits. Discovery draws on train and valid only - the few-shot rows the proposer sees come from train, and its screen fits on train and scores on valid - so test is touched by nothing before the final verdict.

In [6]:
frames = stratified_split(frame, target, valid_size=VALID_SIZE, test_size=TEST_SIZE, seed=SEED)

pd.DataFrame({
    name: {"rows": len(part), "positives": int(part[target].sum()),
           "positive_rate": round(part[target].mean(), 4)}
    for name, part in frames.items()
}).T

,rows,positives,positive_rate
train,1020.0,236.0,0.2314
valid,340.0,79.0,0.2324
test,340.0,79.0,0.2324


## 5. Few-shot example rows

The rows a discovery run prints under every column of its prompt - the only concrete data the proposer ever sees. They are chosen per class by KMeans on train's incumbent columns, one row per cluster (16 per class), so they cover the table rather than its densest region. Batch *r* is shown in round *r*; batch *b* takes the *b*-th closest row of each cluster, so successive rounds see different rows from the same regions.

Saved as the rows themselves - the same columns as `train.csv`, plus `batch` - and read back and cleaned exactly as the splits are, so the prompt shows what the models see.

Only the incumbent measurements - age and the four blood pressures - are clustered as numbers. The other incumbents are clinical codes, so they are one-hot encoded, and a missing value is a level of its own rather than being imputed away.

In [7]:
base, new = resolve_features(frames["train"], cfg.features, cfg.data)
continuous = cfg.discovery.continuous_columns
categorical = ([c for c in base if c not in set(continuous)] if continuous is not None
               else [c for c in cfg.discovery.categorical_columns if c in base])

batches = build_shot_batches(frames["train"], target, columns=base, categorical=categorical,
                             shots=SHOTS, batches=SHOT_BATCHES, seed=SEED)
few_shot = pd.concat([b.assign(batch=i) for i, b in enumerate(batches)], ignore_index=True)
few_shot = few_shot[["batch", *frames["train"].columns]]   # train.csv's columns, plus batch
assert few_shot[id_col].isin(frames["train"][id_col]).all()

print(f"{len(base)} incumbent columns ({len(categorical)} shown as coded categories), "
      f"{len(new)} candidates")
print(f"{len(batches)} batches x {len(batches[0])} rows, from {len(frames['train']):,} train rows")
few_shot.groupby("batch")[target].agg(rows="size", positives="sum").T

101 incumbent columns (96 shown as coded categories), 9 candidates
10 batches x 32 rows, from 1,020 train rows


batch,0,1,2,3,4,5,6,7,8,9
rows,32,32,32,32,32,32,32,32,32,32
positives,16,16,16,16,16,16,16,16,16,16


## 6. Screen sample

The rows discovery's screen fits each proposal on - a sample of **train** - and scores it on - a sample of **valid**. It is the cheap signal the proposer gets back every round; the decision is made later, on the full splits. When `SCREEN_BALANCE` is true each class contributes up to half the rows, so a rare class is kept whole instead of the handful a uniform draw would give. Saved as the rows themselves, in the same format as `train.csv`.

Scoring on valid means the proposer's feedback comes from valid rows, so valid stops being an independent check on what it proposes; test still is. To keep valid independent, draw both samples from disjoint rows of train. `discovery.screen_data: splits` in the config skips these files and uses the whole train and valid splits.

In [8]:
screen = {
    part: build_sample(frames[part], target, SCREEN_SIZE[part],
                       balance=SCREEN_BALANCE, seed=SEED)
    for part in ("train", "valid")
}

pd.DataFrame({
    part: {"rows": len(rows), "positives": int(rows[target].sum()),
           "positive_rate": round(rows[target].mean(), 4)}
    for part, rows in screen.items()
}).T

,rows,positives,positive_rate
train,1000.0,232.0,0.2320
valid,340.0,79.0,0.2324


## 7. Check against the config, then write

`write_splits` refuses to write anything if the tables disagree with `configs/myocardial.yaml`: a candidate the config names but the table lacks, a missing or non-binary target, splits with different columns, or an id in two splits.

In [9]:
written = write_splits(cfg, frames, root=ROOT, mapping=mapping, descriptions=descriptions)
written["few_shot"] = resolve_path(cfg.discovery.few_shot_path, ROOT)
few_shot.to_csv(written["few_shot"], index=False)
for part, rows in screen.items():
    written[f"screen_{part}"] = resolve_path(cfg.discovery.screen_paths[part], ROOT)
    rows.to_csv(written[f"screen_{part}"], index=False)

for name, path in written.items():
    print(f"{name:<13} {path.relative_to(ROOT)}")

train         data/myocardial/train.csv
valid         data/myocardial/valid.csv
test          data/myocardial/test.csv
descriptions  data/myocardial/column_descriptions.json
mapping       data/myocardial/column_mapping.csv
few_shot      data/myocardial/few_shot.csv
screen_train  data/myocardial/screen_train.csv
screen_valid  data/myocardial/screen_valid.csv


Next, from the repo root:

```bash
autofe -c configs/myocardial.yaml                                  # the declared candidates
autofe -c configs/myocardial.yaml --set discovery.enabled=true     # plus LLM-proposed ones
```